# Préparation des données et calcul des rendements

Objectif : construire une table commune de prix ajustés,
puis calculer les rendements quotidiens et mensuels sur 2019–2023.

Les données sont chargées depuis la collecte sauvegardée.
Aucun nouveau téléchargement n'est réalisé.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_dir = Path.cwd().resolve()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

if not (project_dir / "config.py").exists():
    raise FileNotFoundError("Le dossier du projet n'a pas été retrouvé")

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from config import (
    TICKERS,
    PILOT_START,
    PILOT_END,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
)
from src.data_collection import load_data
from src.data_cleaning import prepare_prices, compute_returns

data_folder = RAW_DATA_DIR / "pilote_2019_2023"

raw_data, metadata = load_data(data_folder)

print("Collecte utilisée :", metadata["saved_at_utc"])
print("ETF chargés :", list(raw_data))

Collecte utilisée : 2026-10-23T12:46:50.695516+00:00
ETF chargés : ['SPY', 'IEF', 'GLD', 'BIL']


In [2]:
prices = prepare_prices(raw_data, TICKERS)

print("Dimensions :", prices.shape)
print("Première date :", prices.index.min())
print("Dernière date :", prices.index.max())

display(prices.head())

Dimensions : (1277, 4)
Première date : 2018-12-03 00:00:00
Dernière date : 2023-12-29 00:00:00


,SPY,IEF,GLD,BIL
Date,,,,
2018-12-03,247.790237,83.894737,116.410004,74.463943
2018-12-04,239.761230,84.282104,117.120003,74.472107
2018-12-06,239.397491,84.496368,117.139999,74.488380
2018-12-07,233.834824,84.743622,118.089996,74.496544
2018-12-10,234.278427,84.793106,117.680000,74.504662


In [3]:
daily_returns, monthly_returns = compute_returns(
    prices,
    PILOT_START,
    PILOT_END,
)

# vérif mois présents
expected_months = pd.date_range(
    start=PILOT_START,
    end=PILOT_END,
    freq="ME",
)

if not monthly_returns.index.equals(expected_months):
    raise ValueError("La table ne contient pas tous les mois attendus")

print("Dimensions des rendements quotidiens :", daily_returns.shape)
print("Dimensions des rendements mensuels :", monthly_returns.shape)

print("\nPremiers rendements mensuels, en % :")
display((monthly_returns.head() * 100).round(2))

print("Rendements mensuels manquants :")
display(monthly_returns.isna().sum().to_frame("nombre"))

Dimensions des rendements quotidiens : (1258, 4)
Dimensions des rendements mensuels : (60, 4)

Premiers rendements mensuels, en % :


,SPY,IEF,GLD,BIL
Date,,,,
2019-01-31,8.01,0.65,2.89,0.17
2019-02-28,3.24,-0.53,-0.61,0.18
2019-03-31,1.81,2.66,-1.60,0.17
2019-04-30,4.09,-0.52,-0.66,0.19
2019-05-31,-6.38,3.05,1.76,0.21


Rendements mensuels manquants :


,nombre
SPY,0
IEF,0
GLD,0
BIL,0


In [4]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

prices.to_csv(
    PROCESSED_DATA_DIR / "prices_daily_pilot.csv",
    index_label="Date",
)

daily_returns.to_csv(
    PROCESSED_DATA_DIR / "returns_daily_pilot.csv",
    index_label="Date",
)

monthly_returns.to_csv(
    PROCESSED_DATA_DIR / "returns_monthly_pilot.csv",
    index_label="Date",
)

print("Tables sauvegardées dans :", PROCESSED_DATA_DIR)

Tables sauvegardées dans : C:\Users\DHERALE\Desktop\Projet-Python-ENSAE\data\processed
